# Blind review of the two saved HealthBrowse adapters
Ten educational questions × two audiences × two adapters. These prompts use fixed, source-linked evidence in Buddy chat mode. Select a GPU, enable notebook access to HF_TOKEN, and mount the Drive used for both pilots. Run all cells. Completed outputs are reused after an interruption. Training is never invoked.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, datetime, uuid
SOURCE_COMMIT = '6b25fe2a11ce856746cc55bd7173dccce1e71933'
REPO = Path('/content') / ('healthbrowse-pilot-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--filter=blob:none', '--branch', 'research/healthbrowse-medical-model',
                    'https://github.com/McChandra/medical_chatbot.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if head != SOURCE_COMMIT:
    raise RuntimeError('The pilot checkout does not match the pinned source commit.')
LAB = REPO / 'healthbrowse_model_lab'
os.chdir(LAB)
print('Pinned pilot source:', head)


In [ ]:
# Keep the text-model stack independent of packages preinstalled in the Colab kernel.
import venv
ENV_DIR = Path('/content') / ('healthbrowse-train-' + sys.implementation.cache_tag)
venv.EnvBuilder(with_pip=False).create(ENV_DIR)
LAB_PYTHON = str(ENV_DIR / 'bin/python')
lab_env = os.environ.copy()
lab_env.pop('PYTHONPATH', None)
lab_env['PYTHONNOUSERSITE'] = '1'

def run_lab(*args):
    subprocess.run([LAB_PYTHON, *map(str, args)], cwd=LAB, env=lab_env, check=True)

# Target the venv directly; Colab's system Python may not ship ensurepip.
subprocess.run([sys.executable, '-m', 'pip', '--python', LAB_PYTHON,
                'install', '-e', '.[train,test]'], cwd=LAB, env=lab_env, check=True)
run_lab('-c', """import torch, transformers, accelerate, bitsandbytes
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from transformers.integrations import get_reporting_integration_callbacks
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
print('Trainer imports passed.', 'torch', torch.__version__, 'transformers', transformers.__version__)
""")


In [ ]:
from google.colab import userdata
run_lab('-m', 'hb_model_lab.cli', 'preflight')
run_lab('-c', "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime before training.'")
try:
    lab_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError("Enable notebook access to HF_TOKEN in Colab Secrets.") from exc


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
base = Path('/content/drive/MyDrive/HealthBrowseModelLab')
def choose_run(prefix):
    runs = sorted(p for p in base.glob(prefix+'-*') if (p/'model/adapter/adapter_model.safetensors').is_file())
    if len(runs) == 1:
        return runs[0]
    if not runs:
        raise FileNotFoundError('No completed ' + prefix + ' run in the mounted Drive.')
    print('Available:', ', '.join(p.name for p in runs))
    choice = base / input('Enter the exact run folder name: ').strip()
    if choice not in runs:
        raise ValueError('Choose one of the listed folders.')
    return choice
first = choose_run('ear-pressure')
expanded = choose_run('audience-pairs')
review_dir = expanded / ('blind-ten-v1-' + first.name)
print('Comparing',first.name,'and',expanded.name)


In [ ]:
experiment = r"""
CASES = [
    ('m46533-fs-id1476424', 'How do the two halves of the cerebral cortex communicate?'),
    ('m46502-fs-id2310366', 'How does food keep moving through the digestive tract after I swallow?'),
    ('m46006-fs-id1866078', 'What do bile salts do to fat in food?'),
    ('m46348-fs-id2096817', 'What happens to the soft spaces between a baby’s skull bones as the skull grows?'),
    ('m45988-fs-id1688531', 'Why does evaporating sweat cool the body?'),
    ('m46355-fs-id886176', 'Why can my nose run when I cry?'),
    ('m46408-fs-id2175127', 'What physical change makes the voice deepen during male puberty?'),
    ('m46691-fs-id2473411', 'Where are blood cells made after birth, and how does the location change with age?'),
    ('m46392-fs-id3037989', 'How do ligaments help hold the uterus in position?'),
    ('m46664-fs-id1037269', 'How do the cells that set the heartbeat differ in function from the cells that pump blood?'),
]

import gzip
import json
import sys
from pathlib import Path
import torch
from peft import PeftModel
from hb_model_lab.contracts import Query, validate_answer
from hb_model_lab.data import digest, read_jsonl
from hb_model_lab.prompts import messages
from hb_model_lab.runtime import ModelRunner, adapter_manifest
from hb_model_lab.training import load_config

first, expanded, out = map(Path, sys.argv[1:])
runs = {'original': first, 'expanded': expanded}
corpus_file = Path('data/education-corpus.jsonl.gz')
with gzip.open(corpus_file, 'rt', encoding='utf8') as f:
    passages = {p['id']: p for p in map(json.loads, f)}
test_pages = {passages[i]['url'] for i, _ in CASES}
hashes = {}
configs = {}
for name, folder in runs.items():
    cfg = load_config(folder / 'pilot-config.json')
    configs[name] = cfg
    manifest = adapter_manifest(folder / 'model/adapter', cfg, allow_smoke=True)
    hashes[name] = manifest['adapter_sha256']
    for split in ('train', 'validation'):
        path = folder / 'data' / (split + '.jsonl')
        if digest(path) != manifest[split + '_sha256']:
            raise ValueError('Dataset hash mismatch for ' + name)
        rows = read_jsonl(path)
        if split == 'train' and len(rows) != {'original':2, 'expanded':8}[name]:
            raise ValueError('Unexpected pilot training set: ' + name)
        if any(e['url'] in test_pages for row in rows for e in row['input']['evidence']):
            raise ValueError('A test source page appears in the pilot data.')
if any(configs['original'][k] != configs['expanded'][k] for k in ('base_model','revision')):
    raise ValueError('Adapters need the same base model and revision.')
spec = dict(cases=CASES, adapters=hashes, corpus=digest(corpus_file), base=configs['original']['base_model'],
            revision=configs['original']['revision'], source_commit='6b25fe2a11ce856746cc55bd7173dccce1e71933',
            max_new_tokens=512, do_sample=False, version=1)
private = out / 'coordinator'
private.mkdir(parents=True, exist_ok=True)
spec_text = json.dumps(spec, ensure_ascii=False, sort_keys=True, indent=2)
spec_file = private / 'design.json'
if spec_file.exists() and spec_file.read_text() != spec_text:
    raise ValueError('This results folder belongs to a different comparison.')
spec_file.write_text(spec_text)
runner = None
for name, folder in runs.items():
    pending = []
    for idx, (sid, question) in enumerate(CASES, 1):
        evidence = [{k:passages[sid][k] for k in ('id','title','url','text')}]
        for role in ('general','clinician'):
            row_id = f'Q{idx:02}-{role}'
            path = private / f'{row_id}-{name}.json'
            if not path.exists():
                pending.append((row_id, path, Query(question=question, role=role, mode='chat', topic_kind='knowledge', evidence=evidence)))
    if not pending:
        continue
    if runner is None:
        runner = ModelRunner(folder / 'pilot-config.json', baseline=True)
        runner.model = PeftModel.from_pretrained(runner.model, str(folder / 'model/adapter'), adapter_name=name, is_trainable=False)
    else:
        runner.model.load_adapter(str(folder / 'model/adapter'), adapter_name=name, is_trainable=False)
    runner.model.set_adapter(name)
    runner.model.eval()
    tokenizer = runner.tokenizer
    for row_id, path, query in pending:
        prompt = tokenizer.apply_chat_template(messages(query), tokenize=True, add_generation_prompt=True, return_tensors='pt', return_dict=False).to(runner.model.device)
        with torch.inference_mode():
            output = runner.model.generate(prompt, attention_mask=torch.ones_like(prompt), max_new_tokens=512, do_sample=False,
                      pad_token_id=tokenizer.pad_token_id, eos_token_id=[tokenizer.eos_token_id,tokenizer.convert_tokens_to_ids('<|eot_id|>')])
        raw = tokenizer.decode(output[0,prompt.shape[1]:], skip_special_tokens=True)
        record = dict(case=row_id, role=query.role, question=query.question, evidence=query.evidence[0].model_dump(),raw=raw, model=name)
        try:
            answer = validate_answer(json.loads(raw),query)
            record.update(answer=answer.model_dump(), format_pass=True)
        except ValueError as exc:
            record.update(error=str(exc),format_pass=False)
        temp = path.with_suffix('.partial')
        temp.write_text(json.dumps(record,ensure_ascii=False))
        temp.replace(path)
    print('Completed an inference pass.', flush=True)
"""
run_lab("-c", experiment, first, expanded, review_dir)


In [ ]:
import csv, json, random
from collections import defaultdict
from pathlib import Path

private = review_dir / 'coordinator'
sheet = review_dir / 'blind-review.csv'
key_path = private / 'reveal-key.json'
metrics = ['directness','source_accuracy','audience_fit']
pairs = defaultdict(dict)
for path in sorted(private.glob('Q*.json')):
    rec = json.loads(path.read_text())
    pairs[rec['case']][rec['model']] = rec
if len(pairs) != 20 or any(set(pair) != {'original','expanded'} for pair in pairs.values()):
    raise ValueError('Finish all 40 generations before exporting the sheet.')
if not key_path.exists():
    rng = random.SystemRandom()
    key = {}
    for case in sorted(pairs):
        order = list(pairs[case])
        rng.shuffle(order)
        for label, model in zip('AB',order):
            key[case+'-'+label] = model
    tmp = key_path.with_suffix('.partial')
    tmp.write_text(json.dumps(key))
    tmp.replace(key_path)
key = json.loads(key_path.read_text())
def cell(value):
    return "'"+value if value.lstrip().startswith(('=','+','-','@')) else value
if not sheet.exists():
    tmp = sheet.with_suffix('.partial')
    with tmp.open('w', newline='',encoding='utf-8-sig') as f:
        fields = ['id','audience','question','source','source_url','answer','cited_quotes','format_pass',*metrics,'notes']
        w = csv.DictWriter(f,fieldnames=fields)
        w.writeheader()
        for uid,model in sorted(key.items()):
            r = pairs[uid.rsplit('-',1)[0]][model]
            a = r.get('answer',{})
            claims = a.get('answer',[])
            response = ('\n'.join(c['text'] for c in claims) or a.get('message') or r['raw'])
            quotes = '\n'.join(cit['quote'] for c in claims for cit in c['citations'])
            w.writerow(dict(id=uid,audience=r['role'],question=r['question'],source=r['evidence']['text'],
                            source_url=r['evidence']['url'],answer=cell(response),cited_quotes=cell(quotes),format_pass=r['format_pass']))
    tmp.replace(sheet)
print('Review sheet:',sheet)


## Score before revealing model names
Open blind-review.csv in a spreadsheet. Each General or Clinician question has two anonymous answers, A and B. Their order varies by pair. Give each answer three scores. Give identical answers the same scores.

| Score | Directness | Source accuracy | Audience fit |
|---|---|---|---|
| 0 | Misses the question | Material error, contradiction, or unsupported central claim | Inappropriate for this audience |
| 1 | Answers partly or indirectly | Core claim supported with an unsupported detail or imprecise attribution | Understandable but too technical or lacking precision |
| 2 | Answers the question directly and sufficiently | All material claims supported and correctly attributed | Clear everyday language for General or precise, useful explanation for Clinicians |

Read the source even when format_pass is True. Score readable content in a format failure; score a missing answer 0 throughout. Save the scored CSV back under the same filename in the displayed Drive folder. If you used Google Sheets, export as CSV. Keep the coordinator folder private until scoring finishes. Rerun the last cell to see per-model means for each audience. With multiple reviewers, each should score a separate copy before comparing judgments.


In [ ]:
import csv, json
from collections import defaultdict
key = json.loads((review_dir / 'coordinator/reveal-key.json').read_text())
with (review_dir / 'blind-review.csv').open(encoding='utf-8-sig',newline='') as f:
    scored = list(csv.DictReader(f))
metrics = ['directness','source_accuracy','audience_fit']
if len(scored)!=40 or {r['id'] for r in scored} != set(key):
    raise ValueError('The review sheet must contain each of the 40 original IDs once.')
if not all(r[m].strip() in ('0','1','2') for r in scored for m in metrics):
    print('Scoring is incomplete. Finish the three score columns, then rerun this cell to reveal the means.')
else:
    groups = defaultdict(list)
    for row in scored:
        groups[key[row['id']],row['audience']].append(row)
    for (model,role),items in sorted(groups.items()):
        print(model,role,'n =',len(items),{m:round(sum(int(r[m]) for r in items)/len(items),2) for m in metrics})
